# Handwritten Digit Recognition with MNIST

**Beginner Deep Learning Project** — Recognize handwritten digits (0–9) using a Neural Network built with TensorFlow/Keras.

### What you will learn
1. Load and explore the MNIST dataset  
2. Visualize sample images  
3. Preprocess data (normalize, reshape, one-hot encode)  
4. Build a Neural Network (Sequential API)  
5. Compile, train, and evaluate the model  
6. Plot accuracy/loss curves  
7. Compare predictions vs actual labels  
8. Predict on a custom hand-drawn image  

### Tech stack
Python · TensorFlow/Keras · NumPy · Pandas · Matplotlib · Seaborn · OpenCV

## Setup — Import libraries

**First time only:** open Terminal in this folder and run:

    pip install -r requirements.txt

Then run the cell below.


In [ ]:
# Install packages once (uncomment if needed):
# %pip install -r requirements.txt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

import cv2

np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('TensorFlow version:', tf.__version__)



---
## Step 1 — Load and explore the MNIST dataset

**MNIST** = 70,000 grayscale images of handwritten digits (28×28 pixels), split into:
- **60,000 training** images
- **10,000 test** images

Keras downloads MNIST automatically the first time you run this cell.

In [ ]:
# Load MNIST from Keras (returns training and test sets)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [ ]:


print("=" * 50)
print("MNIST DATASET OVERVIEW")
print("=" * 50)
print(f"Training images shape : {x_train.shape}")   # (60000, 28, 28)
print(f"Training labels shape : {y_train.shape}")   # (60000,)
print(f"Test images shape     : {x_test.shape}")     # (10000, 28, 28)
print(f"Test labels shape     : {y_test.shape}")     # (10000,)
print(f"\nPixel value range     : {x_train.min()} to {x_train.max()}")
print(f"Number of classes     : {len(np.unique(y_train))}  (digits 0-9)")
print(f"Image dimensions      : {x_train.shape[1]} x {x_train.shape[2]} pixels")


In [ ]:
# Quick look with Pandas
df_labels = pd.DataFrame({"label": y_train})
print("\nLabel distribution (first 10 counts):")
print(df_labels["label"].value_counts().sort_index())

---
## Step 2 — Visualize sample images from the dataset

Seeing the data helps you understand what the model must learn.

In [ ]:
# Show 25 random training images in a 5x5 grid
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
fig.suptitle("Sample MNIST Handwritten Digits", fontsize=16, fontweight="bold")

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(x_train))
    ax.imshow(x_train[idx], cmap="gray")
    ax.set_title(f"Label: {y_train[idx]}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Class distribution — all digits should appear roughly equally
plt.figure(figsize=(10, 4))
sns.countplot(x=y_train, palette="viridis")
plt.title("Distribution of Digit Labels in Training Set")
plt.xlabel("Digit (0-9)")
plt.ylabel("Count")
plt.show()

---
## Step 3 — Preprocess the data

Neural networks learn better when inputs are scaled and labels are encoded correctly.

| Step | What we do | Why |
|------|------------|-----|
| **Normalize** | Divide pixels by 255 → range [0, 1] | Faster, stable training |
| **Reshape** | Keep (28, 28) for Flatten layer | Model expects image shape |
| **One-hot encode** | Labels 5 → [0,0,0,0,0,1,0,0,0,0] | Softmax needs 10 outputs |

In [ ]:
# 1. Normalize pixel values from [0, 255] to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


In [ ]:
# 2. Reshape — add channel dimension (optional; Flatten works on 28x28 too)
#    Shape: (samples, 28, 28, 1) for consistency with CNN-style pipelines
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

In [ ]:
# 3. One-hot encode labels (e.g. label 3 -> [0,0,0,1,0,0,0,0,0,0])
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [ ]:
print("After preprocessing:")
print(f"  x_train: {x_train.shape}, pixel range: [{x_train.min():.2f}, {x_train.max():.2f}]")
print(f"  y_train_cat: {y_train_cat.shape}")
print(f"\nExample — label {y_train[0]} one-hot encoded:")
print(y_train_cat[0])

---
## Step 4 & 5 — Build and compile the Neural Network

In [ ]:
# Build the model using Keras Sequential API
model = models.Sequential([
    # Input: 28x28 grayscale image
    layers.Input(shape=(28, 28, 1)),

    # Flatten 28x28 -> 784 features
    layers.Flatten(),

    # Hidden layer 1: 128 neurons, ReLU activation
    layers.Dense(128, activation="relu"),

    # Dropout: randomly disable 20% of neurons during training
    layers.Dropout(0.2),

    # Hidden layer 2: 64 neurons, ReLU activation
    layers.Dense(64, activation="relu"),

    # Output layer: 10 neurons (one per digit), Softmax for probabilities
    layers.Dense(10, activation="softmax"),
])

# Compile — tell Keras how to train
model.compile(
    optimizer="adam",                    # Adam adjusts learning rate automatically
    loss="categorical_crossentropy",     # Multi-class classification loss
    metrics=["accuracy"],                # Track accuracy during training
)

# Show architecture summary
model.summary()

---
## Step 6 — Train the model

- **Epochs**: how many times the model sees the full training set  
- **Batch size**: how many images per weight update  
- **Validation split**: hold out 10% of training data to monitor overfitting

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

# Train the model
history = model.fit(
    x_train,
    y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,   # Use 10% of training data for validation
    verbose=1,
)

print("\nTraining complete!")

---
## Step 7 — Plot training accuracy and loss curves

**What to look for:**
- Training and validation **accuracy** should rise together  
- **Loss** should decrease  
- If validation gets worse while training improves → **overfitting**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curve
axes[0].plot(history.history["accuracy"], label="Training Accuracy", linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy", linewidth=2)
axes[0].set_title("Model Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curve
axes[1].plot(history.history["loss"], label="Training Loss", linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation Loss", linewidth=2)
axes[1].set_title("Model Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 8 — Evaluate on test data

The model has never seen these 10,000 images during training — this is the real performance check.

In [ ]:
# Evaluate on the held-out test set
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=0)

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print("=" * 50)

---
## Step 9 — Predictions: actual vs predicted labels

In [ ]:
# Get predicted classes (highest probability wins)
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Confusion matrix — see which digits get confused
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.title("Confusion Matrix (Actual vs Predicted)")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

In [ ]:
# Visualize predictions: green = correct, red = wrong
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
fig.suptitle("Test Predictions — Green = Correct, Red = Wrong", fontsize=14)

indices = np.random.choice(len(x_test), 20, replace=False)

for ax, idx in zip(axes.flat, indices):
    img = x_test[idx].reshape(28, 28)
    true_label = y_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_probs[idx][pred_label] * 100

    ax.imshow(img, cmap="gray")
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {true_label} | Pred: {pred_label}\n({confidence:.1f}%)",
                 color=color, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Show a few misclassified examples
wrong_idx = np.where(y_pred != y_test)[0]
print(f"Misclassified: {len(wrong_idx)} out of {len(y_test)}")

---
## Step 10 — Predict on a custom image

Use `custom_digit.png` in this folder, or save your own drawing as `my_digit.png`.

MNIST style = **white digit on black background** (use `invert=True` for drawings on white paper).

In [ ]:
def preprocess_custom_image(image_path, invert=True):
    """
    Preprocess a custom image for MNIST prediction.

    Parameters
    ----------
    image_path : str
        Path to PNG/JPG image (hand-drawn or uploaded).
    invert : bool
        If True, invert colors so digit is white on black (MNIST style).

    Returns
    -------
    processed : np.ndarray
        Shape (1, 28, 28, 1), normalized to [0, 1].
    display_img : np.ndarray
        28x28 image for visualization.
    """
    # Read image (grayscale)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    # Resize to MNIST size
    img = cv2.resize(img, (28, 28))

    # Invert if background is white (common for hand-drawn digits)
    if invert:
        img = 255 - img

    # Normalize to [0, 1]
    img = img.astype("float32") / 255.0

    # Shape for model: (batch, height, width, channels)
    processed = img.reshape(1, 28, 28, 1)
    return processed, img




In [ ]:
def predict_custom_digit(image_path, invert=True, show=True):
    """
    Predict the digit in a custom image.

    Returns
    -------
    predicted : int
        Predicted digit (0-9).
    confidence : float
        Confidence percentage.
    """
    processed, display_img = preprocess_custom_image(image_path, invert=invert)
    probs = model.predict(processed, verbose=0)[0]
    predicted = int(np.argmax(probs))
    confidence = float(probs[predicted] * 100)

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        axes[0].imshow(display_img, cmap="gray")
        axes[0].set_title("Preprocessed (28x28)")
        axes[0].axis("off")

        axes[1].bar(range(10), probs * 100, color="steelblue")
        axes[1].set_title(f"Prediction: {predicted} ({confidence:.1f}% confidence)")
        axes[1].set_xlabel("Digit")
        axes[1].set_ylabel("Probability (%)")
        axes[1].set_xticks(range(10))

        plt.tight_layout()
        plt.show()

    return predicted, confidence

print("Custom prediction functions ready.")

In [ ]:
# Demo: save a test MNIST digit as a "custom" image, then predict it
# (In practice, replace this path with YOUR drawn/uploaded image)

sample_idx = 42
sample_img = (x_test[sample_idx].reshape(28, 28) * 255).astype(np.uint8)
# Save as white digit on black background (MNIST style)
cv2.imwrite("custom_digit.png", sample_img)

print("Saved sample image: custom_digit.png")
print(f"(This sample is actually digit '{y_test[sample_idx]}' from the test set)")

# Predict on the saved image
pred, conf = predict_custom_digit("custom_digit.png", invert=False)
print(f"\nPredicted: {pred}  |  Actual: {y_test[sample_idx]}  |  Confidence: {conf:.1f}%")

In [ ]:
# --- YOUR TURN: Predict on your own image ---
# 1. Draw a digit (0-9) in Paint / any app on white background
# 2. Save as 'my_digit.png' in this project folder
# 3. Uncomment and run:

# pred, conf = predict_custom_digit("my_digit.png", invert=True)
# print(f"Your digit is predicted as: {pred} ({conf:.1f}% confidence)")

In [ ]:
# Optional: save the trained model for later use
model.save("mnist_digit_model.keras")
print("Model saved as mnist_digit_model.keras")

---
## Why do Neural Networks work well for image recognition?

1. **Pixels → patterns → digits**  
   Early layers detect simple edges and strokes. Deeper layers combine them into digit shapes.

2. **Automatic feature learning**  
   You don't hand-code rules like "a 8 has two loops." The network learns useful features from data.

3. **Generalization**  
   After seeing thousands of examples, the model recognizes new handwriting it has never seen.

4. **Softmax output**  
   The model outputs a probability for each class, not just a single guess — useful when a digit looks ambiguous.

For MNIST, a simple dense network often reaches **~97–98% accuracy**. Convolutional Neural Networks (CNNs) can push higher — a natural next step after this project.

---
## Common beginner mistakes to avoid

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| **Forgetting to normalize** | Training is slow or unstable | Divide pixels by `255.0` |
| **Wrong label format** | Softmax + sparse labels error | Use `to_categorical()` for 10 classes |
| **Evaluating on training data** | Accuracy looks too good (overfitting) | Always evaluate on **test** set |
| **Not inverting custom images** | Predictions are random | MNIST = white digit on black; invert if you drew black on white |
| **Wrong image size** | Shape mismatch errors | Resize to **28×28** with OpenCV |
| **Too many epochs without checking val loss** | Overfitting | Watch validation curves; use EarlyStopping |
| **Expecting 100% accuracy** | Unrealistic goal | ~98% is excellent for this architecture |
| **Shuffling forgotten** | Slightly worse training | Keras shuffles by default in `fit()` |
| **Using test set for tuning** | Leaking information | Tune on validation; test set only once at the end |

### Next steps to level up
- Try a **CNN** (`Conv2D`, `MaxPooling2D`)  
- Add **EarlyStopping** and **ModelCheckpoint** callbacks  
- Experiment with learning rate and more epochs  
- Deploy with a simple Flask/Gradio drawing app  